In [2]:
import torch
import librosa
import soundfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

print(torch.__version__)
print(torch.cuda.is_available())

from google.colab import files

uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()
uploaded = files.upload()

audio_files = ["2186.wav", "2201.wav", "2213.wav", "2231.wav", "2247.wav"]

notation_files = ["2186.csv", "2201.csv", "2213.csv", "2231.csv", "2247.csv"]

print("Detected audio files:", audio_files)
print("Detected notation files:", notation_files)


class MusicDataset(Dataset):
    def __init__(self, audio_files, label_files):
        self.audio_files = audio_files
        self.label_files = label_files


    def __len__(self):
        return len(self.audio_files)

    def __getitem__(self, index):
        audio_path = self.audio_files[index]
        label_path = self.label_files[index]
        y, sr = librosa.load(audio_path, sr=None)
        D = librosa.stft(y)
        S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
        spectrogram_tensor = torch.tensor(S_db, dtype=torch.float32)

        labels_df=pd.read_csv(label_path)
        numeric_labels=labels_df[["start_time", "end_time", "instrument", "note", "start_beat", "end_beat"]]
        label_tensor=torch.tensor(numeric_labels.values, dtype=torch.float32)
        return spectrogram_tensor, label_tensor

dataset=MusicDataset(audio_files, notation_files)
spectrogram, labels=dataset[0]
print(spectrogram.shape)
print(labels.shape)
print(labels[:5])

loader = DataLoader(dataset, batch_size=1, shuffle=True)

for spectrogram, labels in loader:
    print("Batch spectrogram shape:", spectrogram.shape)
    print("Batch labels shape:", labels.shape)


class CRNN(nn.Module):
  def __init__(self, num_freq_bins=1025, cnn_channels=16, hidden_size=64, num_classes=10):
    super(CRNN, self).__init__()

    self.conv1=nn.Conv2d(in_channels=1, out_channels=cnn_channels, kernel_size=3, padding=1)
    self.relu1=nn.ReLU()
    self.pool1=nn.MaxPool2d(kernel_size=2)

    self.lstm=nn.LSTM(input_size=cnn_channels*(num_freq_bins//2), hidden_size=hidden_size, num_layers=2, batch_first=True)
    self.fc=nn.Linear(hidden_size, num_classes)

  def forward(self, x):
    x=self.conv1(x)
    x=self.relu1(x)
    x=self.pool1(x)

    batch_size, channels, freq, time=x.shape
    x=x.permute(0,3,1,2)
    x=x.reshape(batch_size, time, channels*freq)

    x, _=self.lstm(x)
    x=self.fc(x)

    return x

model=CRNN()
print(model)

2.11.0+cpu
False


Saving 2186.wav to 2186 (1).wav


Saving 2201.wav to 2201 (1).wav


Saving 2213.wav to 2213.wav


Saving 2231.wav to 2231.wav


...

Saving 2186.csv to 2186.csv


Saving 2201.csv to 2201.csv


Saving 2213.csv to 2213 (1).csv


Saving 2231.csv to 2231.csv


Saving 2247.csv to 2247.csv
Detected audio files: ['2186.wav', '2201.wav', '2213.wav', '2231.wav', '2247.wav']
Detected notation files: ['2186.csv', '2201.csv', '2213.csv', '2231.csv', '2247.csv']
torch.Size([1025, 18466])
torch.Size([1635, 6])
tensor([[4.5534e+04, 5.5262e+04, 4.1000e+01, 8.8000e+01, 5.0000e-01, 2.3958e-01],
        [5.5774e+04, 6.0893e+04, 4.1000e+01, 8.7000e+01, 7.5000e-01, 2.3958e-01],
        [6.1406e+04, 7.1646e+04, 4.1000e+01, 8.8000e+01, 1.0000e+00, 4.8958e-01],
        [7.1646e+04, 8.3422e+04, 4.1000e+01, 8.3000e+01, 1.5000e+00, 4.8958e-01],
        [8.3934e+04, 9.6222e+04, 4.1000e+01, 8.0000e+01, 2.0000e+00, 4.8958e-01]])
Batch spectrogram shape: torch.Size([1, 1025, 11566])
Batch labels shape: torch.Size([1, 717, 6])
Batch spectrogram shape: torch.Size([1, 1025, 18466])
Batch labels shape: torch.Size([1, 1635, 6])
Batch spectrogram shape: torch.Size([1, 1025, 9961])
Batch labels shape: torch.Size([1, 946, 6])
Batch spectrogram shape: torch.Size([1, 1025, 6998